# Structured Set Pipeline For CTU Relational

Этот ноутбук строит новый set для одной выбранной базы из `ctu_relational`.

Пайплайн:
- создаёт `local_snapshot.db` для каждого семпла;
- сохраняет все таблицы снапшота в `csv/`;
- генерирует SQL поверх локального снапшота через Codestral;
- поддерживает `aggregate` и `text` режимы;
- держит `20%` текстовых ответов на одну БД;
- строит `golden_cells.sql` для retrieval recall.

Артефакты пишутся в `structured_set/<db_name>/<sample_id>/`.

In [13]:
# Если в окружении не хватает пакетов, раскомментируйте и выполните один раз.
# %pip install -q pandas pymysql python-dotenv sqlglot requests tqdm


In [14]:
import importlib
import structured_set_pipeline

importlib.reload(structured_set_pipeline)

from structured_set_pipeline import PipelineConfig, generate_dataset, generate_single_sample, select_sample_plan


In [15]:
cfg = PipelineConfig(
    db_name="Hockey",
    sample_count=50,
    text_answer_fraction=0.20,
    snapshot_min_rows=60,
    snapshot_max_rows=120,
    output_root="structured_set",
    random_seed=42,
    codestral_api_url="https://api.mistral.ai/v1/chat/completions",
    max_plan_attempts=25,
    max_clause_repairs=5,
    max_sample_retries=200,
    diversity_similarity_threshold=0.78,
    numeric_answer_min_value=7.0,
    numeric_answer_tolerance=3.0,
    enforce_numeric_answer_uniqueness=True,
    drop_empty_predicates=True,
    retry_log_every=5,
    llm_temperature=0.9
)
cfg


PipelineConfig(db_name='Hockey', sample_count=50, text_answer_fraction=0.2, snapshot_min_rows=60, snapshot_max_rows=120, output_root='structured_set', random_seed=42, preview_rows_per_table=3, remote_host='relational.fel.cvut.cz', remote_port=3306, remote_user='guest', remote_password='ctu-relational', codestral_model='codestral-latest', codestral_api_url='https://api.mistral.ai/v1/chat/completions', question_model='mistral-large-latest', question_api_url='https://api.mistral.ai/v1/chat/completions', llm_temperature=0.9, http_timeout=180, max_plan_attempts=25, max_clause_repairs=5, max_sample_retries=200, diversity_similarity_threshold=0.78, diversity_compare_same_join_only=False, numeric_answer_min_value=7.0, numeric_answer_tolerance=3.0, enforce_numeric_answer_uniqueness=True, drop_empty_predicates=True, retry_log_every=5)

In [16]:
sample_plan = select_sample_plan(cfg)
sample_plan

{1: {'sample_index': 1, 'target_join_count': 3, 'answer_mode': 'aggregate'},
 2: {'sample_index': 2, 'target_join_count': 4, 'answer_mode': 'aggregate'},
 3: {'sample_index': 3, 'target_join_count': 3, 'answer_mode': 'aggregate'},
 4: {'sample_index': 4, 'target_join_count': 3, 'answer_mode': 'aggregate'},
 5: {'sample_index': 5, 'target_join_count': 3, 'answer_mode': 'aggregate'},
 6: {'sample_index': 6, 'target_join_count': 3, 'answer_mode': 'aggregate'},
 7: {'sample_index': 7, 'target_join_count': 2, 'answer_mode': 'aggregate'},
 8: {'sample_index': 8, 'target_join_count': 2, 'answer_mode': 'aggregate'},
 9: {'sample_index': 9, 'target_join_count': 3, 'answer_mode': 'aggregate'},
 10: {'sample_index': 10, 'target_join_count': 3, 'answer_mode': 'aggregate'},
 11: {'sample_index': 11, 'target_join_count': 3, 'answer_mode': 'text'},
 12: {'sample_index': 12, 'target_join_count': 2, 'answer_mode': 'aggregate'},
 13: {'sample_index': 13, 'target_join_count': 2, 'answer_mode': 'text'},
 

## Single Sample Debug Run

Сначала обычно удобно прогнать один семпл и посмотреть на структуру артефактов.

In [17]:
# single_result = generate_single_sample(sample_index=1, config=cfg, sample_plan=sample_plan)
# single_result

## Full Run

После sanity-check можно запускать генерацию всех 50 семплов.

In [ ]:
dataset_df = generate_dataset(cfg)
dataset_df.head()

Text-answer sample ids (10 of 50): [11, 13, 22, 23, 36, 41, 42, 46, 47, 50]
Join distribution plan: {3: {'total': 17, 'text': 3, 'aggregate': 14}, 4: {'total': 16, 'text': 3, 'aggregate': 13}, 2: {'total': 17, 'text': 4, 'aggregate': 13}}


Generating Hockey:   0%|          | 0/50 [00:00<?, ?it/s]

sample 1: still searching (1/200); plan loop exhausted after 25 attempts: Attempt 25 failed during block materialization: Final block leaf_1 failed validation: Execution failed on sql 'SELECT playerid, year, tmid, s, g, gdg FROM scoringshootout JOIN master ON scoringshootout.playerid = master.playerid WHERE (birthcountry = 'Canada') 


Generating Hockey:   2%|▏         | 1/50 [23:47<19:25:47, 1427.50s/it]

sample 2: still searching (1/200); plan loop exhausted after 25 attempts: Attempt 25 failed during block materialization: Unable to validate JOIN in block leaf_1: join made the probe empty
sample 2: still searching (5/200); plan loop exhausted after 25 attempts: Attempt 25 failed during block materialization: Unable to validate JOIN in block leaf_1: join made the probe empty
sample 2: still searching (10/200); plan loop exhausted after 25 attempts: Attempt 25 failed during block materialization: Unable to validate JOIN in block leaf_1: join made the probe empty
sample 2: still searching (15/200); plan loop exhausted after 25 attempts: Attempt 25 failed during block materialization: Unable to validate JOIN in block leaf_1: Execution failed on sql 'SELECT * FROM scoring AS s JOIN teams AS t ON s.tmid = t.tmid JOIN abbrev AS a ON t.confid = a.code JOIN teamspost AS tp ON t.tmid = tp.tmid AND s.
sample 2: still searching (20/200); plan loop exhausted after 25 attempts: join 1/4 reject; joi

...

sample 40: still searching (1/200); plan loop exhausted after 25 attempts: Attempt 25 failed during block materialization: Unable to validate JOIN in block leaf_1: join made the probe empty


Generating Hockey:  80%|████████  | 40/50 [9:41:24<2:03:49, 742.96s/it]

sample 41: still searching (1/200); plan loop exhausted after 25 attempts: Attempt 25 failed during block materialization: Base FROM clause failed for block root: None
sample 41: still searching (5/200); plan loop exhausted after 25 attempts: Attempt 25 failed during block materialization: Base FROM clause failed for block root: None


Generating Hockey:  82%|████████▏ | 41/50 [10:11:57<2:40:31, 1070.15s/it]

sample 42: still searching (1/200); Block is missing required field from_clause: {'name': 'leaf_1', 'role': 'leaf', 'select_clause': 'SELECT idgoalie1 AS goalieid FROM combinedshutouts UNION ALL SELECT idgoalie2 AS goalieid FROM combinedshutouts', 'from_clause': '', 'join_clauses': [], 'where_clauses': [], 'group_by_clause': '', 'havi


In [ ]:
cfg = PipelineConfig(
    db_name="lahman_2014",
    sample_count=50,
    text_answer_fraction=0.20,
    snapshot_min_rows=60,
    snapshot_max_rows=120,
    output_root="structured_set",
    random_seed=42,
    codestral_api_url="https://api.mistral.ai/v1/chat/completions",
    max_plan_attempts=25,
    max_clause_repairs=5,
    max_sample_retries=200,
    diversity_similarity_threshold=0.78,
    numeric_answer_min_value=7.0,
    numeric_answer_tolerance=3.0,
    enforce_numeric_answer_uniqueness=True,
    drop_empty_predicates=True,
    retry_log_every=5,
    llm_temperature=0.9
)
sample_plan = select_sample_plan(cfg)
dataset_df = generate_dataset(cfg)

In [ ]:
cfg = PipelineConfig(
    db_name="sakila",
    sample_count=50,
    text_answer_fraction=0.20,
    snapshot_min_rows=60,
    snapshot_max_rows=120,
    output_root="structured_set",
    random_seed=42,
    codestral_api_url="https://api.mistral.ai/v1/chat/completions",
    max_plan_attempts=25,
    max_clause_repairs=5,
    max_sample_retries=200,
    diversity_similarity_threshold=0.78,
    numeric_answer_min_value=7.0,
    numeric_answer_tolerance=3.0,
    enforce_numeric_answer_uniqueness=True,
    drop_empty_predicates=True,
    retry_log_every=5,
    llm_temperature=0.9
)
sample_plan = select_sample_plan(cfg)
dataset_df = generate_dataset(cfg)

In [ ]:
cfg = PipelineConfig(
    db_name="NCAA",
    sample_count=50,
    text_answer_fraction=0.20,
    snapshot_min_rows=60,
    snapshot_max_rows=120,
    output_root="structured_set",
    random_seed=42,
    codestral_api_url="https://api.mistral.ai/v1/chat/completions",
    max_plan_attempts=25,
    max_clause_repairs=5,
    max_sample_retries=200,
    diversity_similarity_threshold=0.78,
    numeric_answer_min_value=7.0,
    numeric_answer_tolerance=3.0,
    enforce_numeric_answer_uniqueness=True,
    drop_empty_predicates=True,
    retry_log_every=5,
    llm_temperature=0.9
)
sample_plan = select_sample_plan(cfg)
dataset_df = generate_dataset(cfg)

In [ ]:
cfg = PipelineConfig(
    db_name="Basketball_men",
    sample_count=50,
    text_answer_fraction=0.20,
    snapshot_min_rows=60,
    snapshot_max_rows=120,
    output_root="structured_set",
    random_seed=42,
    codestral_api_url="https://api.mistral.ai/v1/chat/completions",
    max_plan_attempts=25,
    max_clause_repairs=5,
    max_sample_retries=200,
    diversity_similarity_threshold=0.78,
    numeric_answer_min_value=7.0,
    numeric_answer_tolerance=3.0,
    enforce_numeric_answer_uniqueness=True,
    drop_empty_predicates=True,
    retry_log_every=5,
    llm_temperature=0.9
)
sample_plan = select_sample_plan(cfg)
dataset_df = generate_dataset(cfg)

## Output Structure

Для каждого `sample_id` сохраняются:
- `local_snapshot.db`
- `csv/*.csv`
- `final_query.sql`
- `detail_query.sql`
- `golden_cells.sql`
- `QA.json`
- `metadata.json`

Дополнительно:
- для агрегатных запросов пишется `aggregation_query.sql`;
- для текстовых запросов пишется `query.sql`.